In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import numpy as np
import os
import math
from sklearn.metrics import mean_squared_error, f1_score, confusion_matrix
# AVISO: custom_collate_fn será removido dos DataLoaders abaixo.

# -------------------------------------------------------------------------
# IMPORTACAO DE ARQUIVOS LOCAIS
# -------------------------------------------------------------------------
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

import sys
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Importação CRÍTICA, agora que a função está definida no arquivo anexo
from mm_config_mealey import CONFIG 
# custom_collate_fn NAO E MAIS NECESSARIO E DEVE SER REMOVIDO DO IMPORT
from mm_dataset_mealey import Mavic3Dataset, get_mavic3_datasets 
from mm_transformer_mealey import MultiModalTransformer, NeuralMealyLayer


# =========================================================================
# CONFIGURAÇÕES
# -------------------------------------------------------------------------
DATA_ROOT = CONFIG['data_root']

MAX_GRADIENT_NORM = 1.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

LOSS_WEIGHTS = {
    'pos': 1.0,
    'cls': 1.0,
    'temporal': CONFIG['temporal_loss_weight']
}
print(f"Pesos da Loss: {LOSS_WEIGHTS}")

# =========================================================================
# FUNÇÕES DE TREINAMENTO E VALIDAÇÃO
# (Nao houve mudancas criticas aqui, pois o tratamento do batch esta correto)
# =========================================================================
def train_epoch(model, dataloader, criterion_pos, criterion_cls, optimizer, weights, verbose=False):
    model.train()
    running_loss_pos, running_loss_cls, running_loss_temporal = 0.0, 0.0, 0.0
    total_valid_cls_samples, total_time_steps, total_time_steps_temporal = 0, 0, 0
    all_mealy_states_train = []

    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Treinamento")):
        # T é o comprimento máximo da sequência no batch, B é o batch size
        B, T = batch['image'].shape[:2]
        prev_mealy_state = None
        all_pred_pos, all_pred_class = [], []

        optimizer.zero_grad()
        
        # Loop sequencial para manter o estado Mealy (GRU) entre os passos de tempo
        for t in range(T):
            # As entradas estao corretamente formatadas como (B, 1, D)
            image_t = batch['image'][:, t].to(DEVICE).unsqueeze(1)
            lidar_360_t = batch['lidar_360'][:, t].to(DEVICE).unsqueeze(1)
            livox_avia_t = batch['livox_avia'][:, t].to(DEVICE).unsqueeze(1)
            radar_t = batch['radar'][:, t].to(DEVICE).unsqueeze(1)

            output = model(image_t, lidar_360_t, livox_avia_t, radar_t, prev_mealy_state)
            
            all_pred_pos.append(output['pred_pos']) # (B, 3)
            all_pred_class.append(output['pred_class_seq'].squeeze(1)) # (B, 1)
            
            prev_mealy_state = output['final_mealy_state']
            all_mealy_states_train.append(prev_mealy_state.detach().cpu().numpy())

        # Flatten predictions para cálculo da Loss
        pred_pos_flat = torch.cat(all_pred_pos, dim=0) # (B*T, 3)
        pred_class_flat = torch.cat(all_pred_class, dim=0) # (B*T, 1)
        gt_pos_flat = batch['gt_pos'].reshape(B*T, -1).to(DEVICE)
        gt_class_flat = batch['gt_class'].reshape(B*T).to(DEVICE)

        # Loss posição normalizada (Calculada no batch)
        loss_pos = criterion_pos(pred_pos_flat, gt_pos_flat)
        running_loss_pos += loss_pos.item() * (B*T)
        total_time_steps += (B*T)

        # Loss classificação
        valid_mask = gt_class_flat != -1
        if valid_mask.any():
            valid_gt_class = gt_class_flat[valid_mask].float().unsqueeze(1) 
            valid_pred_class = pred_class_flat[valid_mask]
            loss_cls = criterion_cls(valid_pred_class, valid_gt_class)
            running_loss_cls += loss_cls.item() * valid_gt_class.size(0)
            total_valid_cls_samples += valid_gt_class.size(0)
        else:
            loss_cls = torch.tensor(0.0).to(DEVICE)

        # Loss temporal (smoothness)
        pred_seq = pred_pos_flat.reshape(B, T, 3)
        gt_seq = batch['gt_pos'].to(DEVICE)
        pred_delta = pred_seq[:,1:,:] - pred_seq[:,:-1,:]
        gt_delta = gt_seq[:,1:,:] - gt_seq[:,:-1,:]
        loss_temporal = torch.mean((gt_delta - pred_delta)**2)
        running_loss_temporal += loss_temporal.item() * (B*(T-1))
        total_time_steps_temporal += B*(T-1)

        # Combina perdas
        total_loss = (weights['pos']*loss_pos) + (weights['cls']*loss_cls) + (weights['temporal']*loss_temporal)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRADIENT_NORM)
        optimizer.step()

        if verbose:
            print(f"[Batch {batch_idx+1}/{len(dataloader)}] Loss Pos: {loss_pos.item():.4f}, "
                  f"Cls: {loss_cls.item():.4f}, Temporal: {loss_temporal.item():.4f}")

    # Médias por época
    avg_pos = running_loss_pos / total_time_steps
    avg_cls = running_loss_cls / (total_valid_cls_samples if total_valid_cls_samples>0 else 1)
    avg_temporal = running_loss_temporal / (total_time_steps_temporal if total_time_steps_temporal > 0 else 1)
    return avg_pos, avg_cls, avg_temporal, all_mealy_states_train


def validate_epoch(model, dataloader, criterion_pos, criterion_cls):
    model.eval()
    val_loss_pos, val_loss_cls, val_loss_temporal = 0.0, 0.0, 0.0
    correct_cls, total_cls_samples = 0, 0
    total_time_steps, total_time_steps_temporal = 0, 0
    all_mealy_states_val = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validação"):
            B, T = batch['image'].shape[:2]
            prev_mealy_state = None
            all_pred_pos, all_pred_class = [], []

            for t in range(T):
                image_t = batch['image'][:, t].to(DEVICE).unsqueeze(1)
                lidar_360_t = batch['lidar_360'][:, t].to(DEVICE).unsqueeze(1)
                livox_avia_t = batch['livox_avia'][:, t].to(DEVICE).unsqueeze(1)
                radar_t = batch['radar'][:, t].to(DEVICE).unsqueeze(1)

                output = model(image_t, lidar_360_t, livox_avia_t, radar_t, prev_mealy_state)
                
                prev_mealy_state = output['final_mealy_state']
                all_mealy_states_val.append(prev_mealy_state.detach().cpu().numpy())

                all_pred_pos.append(output['pred_pos'])
                all_pred_class.append(output['pred_class_seq'].squeeze(1)) # (B, 1)

            pred_pos_flat = torch.cat(all_pred_pos, dim=0)
            pred_class_flat = torch.cat(all_pred_class, dim=0)
            gt_pos_flat = batch['gt_pos'].reshape(B*T, -1).to(DEVICE)
            gt_class_flat = batch['gt_class'].reshape(B*T).to(DEVICE)

            # Loss
            loss_pos = criterion_pos(pred_pos_flat, gt_pos_flat)
            val_loss_pos += loss_pos.item()*(B*T)
            total_time_steps += B*T

            valid_mask = gt_class_flat != -1
            if valid_mask.any():
                valid_gt_class = gt_class_flat[valid_mask].float().unsqueeze(1)
                valid_pred_class = pred_class_flat[valid_mask]
                loss_cls = criterion_cls(valid_pred_class, valid_gt_class)
                val_loss_cls += loss_cls.item()*valid_gt_class.size(0)

                predicted = (valid_pred_class>0).long().squeeze(1)
                if predicted.ndim > 1: # Garante shape (N_valid)
                    predicted = predicted.squeeze(1) 
                correct_cls += (predicted == gt_class_flat[valid_mask]).sum().item()
                total_cls_samples += valid_gt_class.size(0)

            # Loss temporal
            pred_seq = pred_pos_flat.reshape(B, T, 3)
            gt_seq = batch['gt_pos'].to(DEVICE)
            pred_delta = pred_seq[:,1:,:] - pred_seq[:,:-1,:]
            gt_delta = gt_seq[:,1:,:] - gt_seq[:,:-1,:]
            loss_temporal = torch.mean((gt_delta - pred_delta)**2)
            val_loss_temporal += loss_temporal.item() * (B*(T-1))
            total_time_steps_temporal += B*(T-1)

    avg_pos = val_loss_pos / total_time_steps
    avg_cls = val_loss_cls / (total_cls_samples if total_cls_samples>0 else 1)
    avg_temporal = val_loss_temporal / (total_time_steps_temporal if total_time_steps_temporal > 0 else 1)
    accuracy_cls = (correct_cls / total_cls_samples)*100 if total_cls_samples>0 else 0.0

    print(f"[Validação] Loss Pos: {avg_pos:.4f}, Cls: {avg_cls:.4f}, Temporal: {avg_temporal:.4f}, Accuracy: {accuracy_cls:.2f}%")
    return avg_pos, avg_cls, avg_temporal, accuracy_cls, all_mealy_states_val


# =========================================================================
# TESTE FINAL
# =========================================================================
def test_pipeline(model, dataloader):
    model.eval()
    all_gt_pos, all_pred_pos, all_gt_class, all_pred_class, all_mealy_states_test = [], [], [], [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Teste"):
            B, T = batch['image'].shape[:2]
            prev_mealy_state = None

            for t in range(T):
                image_t = batch['image'][:, t].to(DEVICE).unsqueeze(1)
                lidar_360_t = batch['lidar_360'][:, t].to(DEVICE).unsqueeze(1)
                livox_avia_t = batch['livox_avia'][:, t].to(DEVICE).unsqueeze(1)
                radar_t = batch['radar'][:, t].to(DEVICE).unsqueeze(1)

                output = model(image_t, lidar_360_t, livox_avia_t, radar_t, prev_mealy_state)
                
                prev_mealy_state = output['final_mealy_state']
                all_mealy_states_test.append(prev_mealy_state.detach().cpu().numpy())

                all_gt_pos.append(batch['gt_pos'][:, t, :].cpu().numpy())
                all_pred_pos.append(output['pred_pos'].cpu().numpy())
                all_gt_class.append(batch['gt_class'][:, t].cpu().numpy())
                
                pred_class_logits = output['pred_class_seq'].squeeze(1).cpu().numpy()
                all_pred_class.append((pred_class_logits > 0).astype(int))

    all_gt_pos = np.concatenate(all_gt_pos, axis=0)
    all_pred_pos = np.concatenate(all_pred_pos, axis=0)
    all_gt_class = np.concatenate(all_gt_class, axis=0)
    all_pred_class = np.concatenate(all_pred_class, axis=0)
    
    if all_pred_class.ndim > 1 and all_pred_class.shape[1] == 1:
        all_pred_class = all_pred_class.squeeze(1)

    # Remove amostras de classe inválidas para as métricas de classificação
    valid_mask_cls = all_gt_class != -1
    valid_gt_class = all_gt_class[valid_mask_cls]
    valid_pred_class = all_pred_class[valid_mask_cls]

    rmse_pos = math.sqrt(mean_squared_error(all_gt_pos, all_pred_pos))
    smoothness_error = np.mean(np.linalg.norm(np.diff(all_pred_pos, axis=0) - np.diff(all_gt_pos, axis=0), axis=1))
    
    # Se nao houver amostras validas, evita erros
    if valid_gt_class.size > 0:
        f1_cls = f1_score(valid_gt_class, valid_pred_class, average='weighted', zero_division=0)
        conf_matrix = confusion_matrix(valid_gt_class, valid_pred_class)
    else:
        f1_cls = 0.0
        conf_matrix = np.array([[0,0],[0,0]]) 


    print(f"=== Teste Final ===")
    print(f"RMSE Pos: {rmse_pos:.4f}")
    print(f"Erro Suavidade Temporal: {smoothness_error:.4f}")
    print(f"F1-Score Cls: {f1_cls:.4f}")
    print("Matriz de Confusão (apenas classes válidas):")
    print(conf_matrix)
    print(f"Total de estados Mealy registrados: {len(all_mealy_states_test)}")

    return {
        'rmse_pos': rmse_pos,
        'smoothness_error': smoothness_error,
        'f1_score_cls': f1_cls,
        'conf_matrix': conf_matrix,
        'mealy_states': all_mealy_states_test
    }


# =========================================================================
# MAIN (AJUSTADO)
# =========================================================================
def main_adjusted():
    try:
        datasets = get_mavic3_datasets(DATA_ROOT, CONFIG)
        train_dataset, val_dataset, test_dataset = datasets['train'], datasets['val'], datasets['test']

        # AJUSTE CRÍTICO: Removendo o collate_fn customizado, pois as PCLs agora são de tamanho fixo (9).
        train_loader = DataLoader(
            train_dataset, 
            batch_size=CONFIG["batch_size"], 
            shuffle=True, 
            num_workers=CONFIG["num_workers"], 
            pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=CONFIG["batch_size"], 
            shuffle=False, 
            num_workers=CONFIG["num_workers"], 
            pin_memory=True
        )
        test_loader = DataLoader(
            test_dataset, 
            batch_size=CONFIG["batch_size"], 
            shuffle=False, 
            num_workers=CONFIG["num_workers"], 
            pin_memory=True
        )

    except Exception as e:
        print(f"Erro ao carregar datasets: {e}")
        return

    model = MultiModalTransformer(CONFIG).to(DEVICE)
    criterion_pos = nn.MSELoss()
    criterion_cls = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CONFIG["learning_rate"]*5, steps_per_epoch=len(train_loader), epochs=CONFIG["epochs"]
    )

    best_val_loss = float('inf')
    model_path = 'best_mm_transformer_mealey_model.pth'

    for epoch in range(CONFIG["epochs"]):
        print(f"\n=== Época {epoch+1}/{CONFIG['epochs']} ===")
        train_loss_pos, train_loss_cls, train_loss_temporal, _ = train_epoch(
            model, train_loader, criterion_pos, criterion_cls, optimizer, LOSS_WEIGHTS, verbose=False
        )
        print(f"[Treino] Pos: {train_loss_pos:.4f}, Cls: {train_loss_cls:.4f}, Temporal: {train_loss_temporal:.4f}")

        val_loss_pos, val_loss_cls, val_loss_temporal, val_acc_cls, _ = validate_epoch(model, val_loader, criterion_pos, criterion_cls)
        total_val_loss = LOSS_WEIGHTS['pos']*val_loss_pos + LOSS_WEIGHTS['cls']*val_loss_cls + LOSS_WEIGHTS['temporal']*val_loss_temporal

        if total_val_loss < best_val_loss:
            best_val_loss = total_val_loss
            torch.save(model.state_dict(), model_path)
            print(f"Novo melhor modelo salvo com Loss total: {best_val_loss:.4f}")

        scheduler.step()

    # Teste final
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    test_results = test_pipeline(model, test_loader)
    print("Número de estados Mealy registrados no teste:", len(test_results['mealy_states']))


if __name__ == "__main__":
    main_adjusted()

Usando dispositivo: cpu
Pesos da Loss: {'pos': 1.0, 'cls': 1.0, 'temporal': 0.5}

Divisao do Dataset (Total: 833 amostras):
 - Treino: 665
 - Validacao: 84
 - Teste: 84


Pre-carregando GT para normalização: 100%|██████████| 665/665 [00:00<00:00, 7637.83it/s]

Normalizador de Posição inicializado. Média: 6.7959, Std: 4.6079
Dataset inicializado com 665 amostras.
Dataset inicializado com 84 amostras.
Dataset inicializado com 84 amostras.



=== Época 1/10 ===


Treinamento:   0%|          | 0/9 [00:00<?, ?it/s]c:\Users\Micro\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Treinamento: 100%|██████████| 9/9 [00:29<00:00,  3.24s/it]


...

Validação: 100%|██████████| 1/1 [00:06<00:00,  6.60s/it]


[Validação] Loss Pos: 0.9237, Cls: 0.5358, Temporal: 1.8613, Accuracy: 100.00%
Novo melhor modelo salvo com Loss total: 2.3901


Teste: 100%|██████████| 1/1 [00:08<00:00,  8.59s/it]

=== Teste Final ===
RMSE Pos: 1.0011
Erro Suavidade Temporal: 2.1352
F1-Score Cls: 1.0000
Matriz de Confusão (apenas classes válidas):
[[80]]
Total de estados Mealy registrados: 10
Número de estados Mealy registrados no teste: 10



c:\Users\Micro\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
